# From Plan to PR: Building a FastAPI Feature with BAND

Companion notebook for [the complete To Data & Beyond tutorial](https://todatabeyond.com/blog/from-plan-to-pr-building-a-fastapi-feature-with-band). View the [maintained notebook on GitHub](https://github.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/FastAPI_Feature_with_BAND.ipynb).


## Before you begin

This companion is intentionally documentation-first. Run the commands in a local terminal and review the BAND configuration before connecting coding agents to a repository.

This notebook intentionally contains no saved execution outputs or credentials.


How I used Claude Code and Codex in a shared BAND workflow to plan, review, and implement a real backend feature

Most devs I’ve talked to lately are running two or three coding agents at once. Claude Code for architecture and planning. Codex for writing or reviewing code. Cursor or whatever IDE agent for quick edits without leaving the editor.

Each tool is fine on its own. The gluing-together part isn’t. You ask Claude to design a feature, paste the output into Codex for a second pass, bring the feedback back, and somewhere in there, you’re also trying to keep the repo, the open files, and half-made decisions aligned across three tabs. You end up being the integration layer, and the whole thing runs at copy-paste speed.

BAND is trying to fix that. Agents share a repo and a chat, and they hand work off through structured roles, so context doesn’t have to live in your head or on your clipboard. “More agents” isn’t the interesting part. What’s interesting is that they can actually work together without you ferrying state between them.

I wanted to see if any of that would help with a real task, so I built JWT auth with role-based access control in a small FastAPI app. Login flow, tokens, protected routes, one admin-only endpoint, and enough structure to test it properly. Claude Code planned it. Codex reviewed.

BAND held the shared state. The rest of the post is what I actually saw, which parts earned their keep, which felt like a ceremony, and whether running the same agents through a coordination layer is meaningfully different from running them by hand.

## Table of Contents

---

## 1. The problem with multi-agent coding workflows

Running more than one coding agent sounds efficient until you try it on something real. One tool plans better, another reviews better, and a third lives in the IDE for small edits. All fine. They just don’t share anything — not context, not files, not a sense of what the other one already said.

So you become the bus between them. Plan gets generated in Claude. You paste it into Codex to poke holes. You bring the notes back. And somewhere in there, you’re also tracking which version of the plan is current, what each agent has actually seen of the repo, and whether the review still applies after you’ve touched three files. The outputs are usually good. It’s what happens in between that drags.

Auth is where it really shows. You can’t just ask one model for a login endpoint, ask another to review it, and call it done. Token handling, route guards, permissions, and tests all have to agree with each other, and when they quietly disagree, you often don’t find out until something breaks in a way the tests didn’t cover. Spread that design across three tabs, and you’re the one keeping it coherent.

The annoying part is that the models themselves aren’t really the limit anymore. The planner is good. The reviewer catches real things. The handoff between them is what’s weak. You re-paste the same context, re-attach the same files, re-explain the same constraints, and a decent chunk of your day is just moving state around.

That’s what I actually wanted to test. Not whether I could squeeze more out of a single agent, but what it feels like when the agents share a repo, a chat, and a view of the code, and hand work off to each other without routing through me first.

---

## 2. What are we building?

I needed something small enough to walk through in one article, but big enough that planning and review would actually earn their keep. JWT auth with role-based access control in a FastAPI app was about right.

Auth is a good stress test because it pulls in app design, security, and a bunch of boring mechanics all at the same time. Even in a tiny FastAPI project, “add login” ends up being a pile of decisions: how tokens are signed, how long they live, what the refresh flow looks like, how routes are guarded, how roles get checked, and how any of this gets tested. Plenty to argue about at planning. Plenty to get wrong if nobody pushes back on it.

The scope I gave myself:

- a login endpoint
- JWT access tokens
- a refresh token flow
- protected routes that require an authenticated user
- An admin-only route enforced by role check
- a basic test setup covering the auth flow and the access control behavior

That’s enough for a planner to break into phases, and enough for a reviewer to find real gaps in. It’s also the kind of feature where you notice if you skipped the plan-and-review step — you end up patching security holes you would have caught earlier if someone had looked at the design on paper.

Not a toy example, not a whole product. Just enough moving parts, enough decisions that actually matter, and enough security surface to make review feel less optional.

---

## 3. Installing Claude Code, Codex, and BAND

For this workflow, I used Claude Code as the planner, Codex as the reviewer, and BAND as the shared coordination layer between them. That role split matches BAND’s coding-agents setup, which presents a planner-reviewer workflow in a shared repo and chat environment rather than a series of disconnected prompts.

Before setting everything up, I needed a few prerequisites:

- Python 3.11+
- the uv package manager
- Node.js 20+ for the Claude Code side
- the Claude Code CLI
- the Codex CLI
- external agents created on the BAND platform
- API credentials for the Claude and Codex sides of the workflow.

### 3.1. Create a local project and install the SDK

The cleanest working install path is to use the published PyPI package instead of installing directly from the GitHub repo. The PyPI package is band-sdk, and it exposes extras for both Claude Agent SDK and Codex support.

Start by creating a project and installing the SDK:


**Command or configuration**

```bash
mkdir band-agent-demo && cd band-agent-demo
uv init
uv add "band-sdk[claude-sdk,codex]"
```


This avoids the GitHub submodule issue you hit earlier and is the safer setup path for a reader following the article. The package metadata also confirms that Python 3.11 or newer is required.

### 3.2. Install Claude Code & Codex

You can install Claude Code with the following command:


**Command or configuration**

```bash
curl -fsSL https://claude.ai/install.sh | bash
```


Once it is installed, you can start with this command:


**Command or configuration**

```bash
claude
```


You can install Codex CLI with the following command:


**Command or configuration**

```bash
npm install -g @openai/codex
```


Once it is installed, you can start with this command:


**Command or configuration**

```bash
codex
```


### 3.3. Create your agents on BAND

Next, create the external agents inside the BAND platform. For this article, the setup uses two agents:

- a planner agent for Claude Code
- a reviewer agent for Codex.

When each external agent is created, you need to save two values:

- the agent UUID
- the API key

The SDK docs say the API key is shown at creation time, and the agent UUID is available from the agent details page. These are the credentials your local agent processes that will be used to connect back to BAND.

### 3.4. Add environment variables

Create a .env file for the platform endpoints and model credentials. The SDK examples still use THENVOI_* variable names:


**Command or configuration**

```ini
THENVOI_REST_URL=https://app.thenvoi.com/
THENVOI_WS_URL=wss://app.thenvoi.com/api/v1/socket/websocket
ANTHROPIC_API_KEY=sk-ant-...
OPENAI_API_KEY=sk-...
```


Even though the platform branding is now BAND, using the Thenvoi-style variable names is consistent with the current package examples and adapter docs.

### 3.5. Store the agent credentials

Create an agent_config.yaml file and add both agents:


**Command or configuration**

```yaml
planner:
  agent_id: "<planner-agent-uuid>"
  api_key: "<planner-agent-api-key>"
reviewer:
  agent_id: "<reviewer-agent-uuid>"
  api_key: "<reviewer-agent-api-key>"
```


The SDK docs show this same pattern for loading credentials from agent_config.yaml, and they note that the config file is intended to store the agent_id and api_key for each external agent.

### 3.6. How Claude Code and Codex fit into this setup

At this point, the workflow becomes more specific than a generic “AI agent” example.

Claude Code acts as the planner. The SDK supports this through the ClaudeSDKAdapter, which is described as a production-ready Claude Agent SDK integration. The package docs also show a Claude SDK example using ClaudeSDKAdapter and a Claude model.

Codex acts as the reviewer. The SDK supports this through the CodexAdapter and CodexAdapterConfig, including runtime controls such as transport, approval handling, and working directory. The package docs explicitly list Codex App-Server support as production-ready and provide a Codex adapter example.

BAND’s coding-agents page describes the same overall pattern at a higher level: a planner writes the plan, a reviewer critiques it, and both work in the same shared repo and chatroom.

### 3.7. Why BAND matters in this workflow

The main value of BAND here is not simply that it lets me connect multiple agents. The more important part is that it gives those agents a shared coordination layer. On the coding-agents page, BAND describes this as a setup in which agents share the same repo, files, and chatroom, and can coordinate directly rather than relying on the developer to shuttle outputs between tools.

That is what makes this setup useful for a FastAPI feature like JWT auth and RBAC. Claude Code can produce the implementation plan, Codex can review that plan against the same project context, and the handoff happens in one connected workflow rather than across separate tools and pasted messages.

---

## 4. Planning the FastAPI Authentication Feature

With the setup in place, the first real step was planning the feature. In this workflow, the planning did not start with me pasting a long one-off prompt into Claude Code manually.

Instead, the instruction layer was split into two parts: persistent role instructions on the BAND side, and a task-specific request sent in the BAND chatroom. BAND’s coding-agents setup explicitly uses role prompt files such as prompts/planner.md and prompts/reviewer.md, and it describes those files as defining behavior, workspace conventions, communication rules, and handoff protocols.

That distinction matters. The reusable behavior for the planner belongs in the role prompt, not in an ad hoc task message. In practice, that means the planner can already know things like:

- It should produce a phased implementation plan,
- It should write the output to the plan.md,
- It should include risks, deliverables, and acceptance criteria.
- It should hand the work off to the reviewer when the plan is ready. BAND describes the built-in planner role in almost exactly these terms and notes that the planner owns /workspace/notes/plan.md.

The actual task then starts from the BAND chatroom with a direct request to the planner. BAND’s own example uses a message like @Planner Design an auth system for our API, after which the planner writes the plan, mentions the reviewer, and waits for feedback. The same page explains that when a message like this is sent, the planner reads the generated codebase context, writes a phased plan to /workspace/notes/plan.md, then hands it off for review.

For this FastAPI feature, the task-level message would look something like this:


**Prompt**

```prompt
@Planner Design a JWT authentication and RBAC feature for this FastAPI API.
```


That short message works because the heavy lifting has already been moved into the role setup. The planner does not need to be re-told on every run how to behave, where to write the output, or when to hand off to the reviewer. Those conventions are already defined by the BAND-side role prompt system.

In this article’s workflow, Claude Code acts as the planner through BAND’s Claude Agent SDK integration, while Codex acts as the reviewer through BAND’s Codex adapter. BAND documents both of these as supported coding-agent paths, and its examples also show optional custom_section fields on both adapters for light behavior tuning.

I would treat those adapter-level instructions as secondary. They are useful for short preferences such as “focus on maintainable designs” or “keep changes minimal,” but the main planning workflow should still be defined by the role prompt files and the BAND chat handoff.

Once the request is sent, the planning phase becomes much more structured than a normal one-shot prompt. Instead of jumping straight into implementation, Claude Code produces a concrete design artifact first. For this feature, the plan needed to cover the main pieces of the work:

- a login endpoint,
- JWT access token generation,
- a refresh token flow,
- protected routes for authenticated users,
- an admin-only route,
- and a basic testing plan.

That structure is exactly why planning is useful here. Authentication is one of those features where missing details tend to show up later as rework: incomplete token lifecycle design, weak route protection, unclear permission boundaries, or missing test coverage.

By forcing the workflow to begin with a plan written to a shared file, the next step becomes much easier: the reviewer can challenge the design before those problems turn into implementation mistakes. BAND’s model is explicit about this split: chat is for coordination, files are for content.

So in this workflow, the planning step is not just “ask Claude Code what to build.” It is a small system:

- BAND role prompts define how the planner should behave,
- BAND chatroom carries the task-specific request.
- Claude Code generates the structured plan,
- This plan becomes the artifact that moves into review next.

---

## 5. Reviewing the Plan Before Writing Code

Once the initial plan was written, the next step was to review it before moving anywhere near implementation. This is one of the most useful parts of the workflow. In BAND’s coding-agents setup, the reviewer is not just reacting to pasted text from another tool.

The reviewer works in the same shared environment, reads the same repository context, checks the plan artifact directly, and writes feedback into a shared review file. BAND’s own example describes this as a loop where the planner writes plan.md, the reviewer critiques it in review.md, and the cycle repeats until the design is ready.

For this workflow, Codex acted as the reviewer. Its job was not to rewrite the feature from scratch, but to pressure-test the proposed design before implementation started. BAND describes the reviewer role as one that cross-references plans against the source code and categorizes feedback as Critical, Risk, Gap, or Suggestion. That classification is useful because it keeps the review focused and makes the next revision step clearer.

This matters because a planning artifact can look complete while still missing important details. For a FastAPI authentication feature, the review should focus on questions such as:

- Is the refresh token flow clearly defined?
- Are authentication and authorization separated cleanly?
- Is the route protection strategy consistent with FastAPI’s dependency model?
- Are there missing security controls around token issuance or refresh?
- Does the plan include a realistic testing strategy?

A good reviewer should catch these issues before they become code. That is the difference between using review as a quality gate and using it only as a cleanup after implementation.

In the BAND workflow, the handoff to the reviewer happens directly inside the shared chatroom. The planner writes the plan, mentions the reviewer, and waits.

The reviewer then reads the plan file, checks the relevant repository context, writes structured findings to review.md, and posts a verdict back into the same coordination flow. BAND explicitly describes this handoff model and notes that the reviewer can open relevant source files in the shared workspace before returning a decision.

In practical terms, the reviewer message might look something like this after inspecting the first version of the plan:


**Prompt**

```prompt
Changes requested. Missing refresh token rotation [Critical], no rate limiting on token endpoints [Risk], and incomplete test coverage for protected routes [Gap].
```


That kind of output is much more useful than a vague “this looks good” or a generic warning about security. It gives the planner concrete issues to address before implementation starts. It also makes the workflow easier to follow because the feedback is tied to categories and stored in a shared artifact rather than disappearing inside a one-off chat response.

This is where the multi-agent setup starts to feel practical rather than decorative. The value is not simply that one model plans and another reviews. The real value is that the review happens against the same project context and in the same workflow.

I do not need to manually copy the plan into another tool, restate the feature, or re-explain the repository structure. BAND’s setup is built specifically around that shared repo, shared notes, and shared chat model.

For this FastAPI feature, that meant the review stage could focus on the parts that usually cause trouble later:

- token lifecycle design,
- route protection boundaries,
- role enforcement logic,
- and testing gaps.

By the end of this step, the goal was not to have code yet. The goal was to have a plan that had already survived one serious round of scrutiny. That makes the next step much stronger, because implementation begins from an approved design rather than from a first draft.

---

## 6. Revising the Plan and Moving to Implementation

Once the review came back, the workflow moved into revision rather than straight into coding. That is an important distinction. In BAND’s coding-agents model, the reviewer does not simply leave comments and disappear.

The planner is expected to update plan.md, address the issues recorded in review.md, and hand the revised plan back for another pass if needed. The product page describes exactly that loop: the planner writes the plan, the reviewer requests changes, the planner updates the plan, and the cycle repeats until the design is approved.

For this FastAPI feature, that meant the first version of the plan was treated as a draft, not as something implementation-ready by default. If the reviewer flagged a missing refresh token flow as Critical, weak rate limiting as a Risk, or incomplete test coverage as a Gap, the planner needed to incorporate those changes directly into the plan before any implementation work started. BAND’s built-in role descriptions make that split very explicit: the planner owns the plan.md, while the reviewer owns the review.md.

This is one of the places where the workflow feels more disciplined than a normal “ask one model, then ask another” setup. The handoff is structured:

- The planner revises the design
- The reviewer checks whether the changes actually resolve the concerns,
- And only then is the feature handed back as ready for implementation. BAND provides the shared chatroom, where the planner updates the plan, mentions the reviewer again, and waits for the verdict.

A revised handoff message in this stage might look something like this:


**Prompt**

```prompt
Updated plan.md — added refresh token rotation, rate limiting requirements, and a test matrix for protected and admin-only routes. @Reviewer
```


That kind of revision matters because it turns the plan into a real implementation artifact rather than a rough conversation summary. For an authentication feature, small omissions at the planning stage can easily create larger problems later.

If the token lifecycle is underspecified, if the admin checks are not clearly separated from basic authentication, or if the testing strategy is vague, the implementation stage tends to inherit that ambiguity.

BAND’s workflow is designed to reduce exactly that kind of drift. The platform describes the planner role as producing structured plans with phases, deliverables, acceptance criteria, risks, and open questions, while the reviewer cross-references the plan against the source code and categorizes its findings. That gives the revision step a clear purpose: close the gaps before implementation begins.

By the time this step is complete, the goal is not perfection. The goal is to have a plan that is coherent enough that implementation can proceed without immediately reopening the design discussion.

Once the reviewer approves, the workflow changes meaningfully: the feature is no longer a proposal being debated, but an approved design ready to be turned into code. BAND’s own example reflects that exact transition, ending with the reviewer approving the revised plan and handing it back as ready for implementation.

That makes the next stage much stronger. Instead of asking an agent to “build JWT auth in FastAPI” from scratch, the implementation starts from a reviewed design with clearer boundaries, clearer requirements, and fewer hidden assumptions. In other words, the revision step is what turns the planning phase from a nice idea into something solid enough to build on.

---

## 7. Implementing the Feature in FastAPI

Once the plan had been reviewed and approved, the workflow moved into implementation. At this point, the value of the earlier steps became clear. Instead of asking an agent to generate JWT authentication from scratch, the implementation started from a design that had already been structured, challenged, and revised. For a feature like authentication, that matters because small design gaps can quickly turn into security issues or unnecessary rework.

For this FastAPI feature, the implementation naturally broke into five main parts:

- Authentication endpoints,
- Token creation and validation,
- Protected routes,
- Role-based access control,
- and tests.

Because these pieces had already been defined during planning, implementation felt less like open-ended exploration and more like executing a reviewed design.

### 7.1. Creating the authentication endpoints

The first part was adding the core authentication endpoints. At a minimum, that meant a login route for verifying user credentials and issuing an access token, along with a refresh route for generating a new access token when needed.

The important point here is that these endpoints should not be treated as isolated route handlers. They depend on a broader authentication layer that defines how tokens are created, what claims they include, and how expiration is handled. That structure is much easier to implement cleanly when it has already been thought through during planning.

### 7.2. Validating tokens and protecting routes

The next step was protecting routes that require an authenticated user. In FastAPI, this is usually handled through dependencies that validate the incoming token and resolve the current user before the request reaches the endpoint logic.

This separation keeps the implementation cleaner. Authentication focuses on identifying the user, while authorization focuses on what that user is allowed to do. Keeping those concerns distinct makes the system easier to reason about and reduces the temptation to scatter token-parsing logic across multiple endpoints.

### 7.3. Adding role-based access control

After basic authentication was in place, the next layer was authorization. Since the feature included an admin-only route, the implementation needed a role check on top of simple user authentication.

This is where the earlier review step helped a lot. A weak design often stops at “protect the route” without defining where role checks should live or how they should be enforced consistently. By the time implementation started here, those decisions had already been clarified. That made it much easier to add an admin-only access layer without mixing it awkwardly into the rest of the authentication flow.

### 7.4. Adding tests as part of the feature

The last part of the implementation was testing. For authentication and access control, tests should be part of the feature itself rather than something added later if there is time.

The core cases were straightforward:

- successful and failed login,
- access to protected routes with and without a valid token,
- access to admin-only routes with and without the required role,
- and refresh-token behavior.

Because testing had already been included in the reviewed plan, it felt like a built-in part of implementation rather than an afterthought.

The main difference in this stage was not that implementation became automatic. The difference was that coding started from a reviewed design instead of from a rough prompt. The planner had already broken the feature into parts, the reviewer had already challenged weak spots, and the final plan had already established the boundaries of the work.

That made implementation more focused. Instead of repeatedly stopping to revisit design decisions, the workflow could concentrate on turning the approved plan into actual FastAPI components.

---

## 8. What Worked Well in This Workflow

The strongest part of this workflow was the separation of responsibilities. Instead of asking one agent to do everything, the work was split into planning, review, and implementation.

That made the process feel more structured and easier to follow. Claude Code could focus on designing the feature, Codex could focus on challenging the design, and implementation could start only after the plan had been improved.

Another thing that worked well was the use of shared artifacts. The plan and the review were not buried inside disconnected chat messages. They existed as concrete outputs in the workflow, which made it much easier to understand what had already been decided, what still needed to be revised, and when the feature was ready to move forward. That is a small detail, but it makes a big difference once the task is more complex than a single prompt.

The review step was also more useful than a typical second opinion. In many multi-tool workflows, review happens in a weak form: one model gives feedback on pasted text with only partial context.

Here, the review was part of the same working environment, which made it easier to push on missing details before implementation began. For a feature like JWT auth and RBAC, that is exactly where review adds the most value. It is much better to catch weak token handling or incomplete access rules in the design stage than after the routes have already been implemented.

The workflow also reduced a lot of the manual coordination overhead. Normally, using multiple coding agents means keeping track of who saw what, which output is the latest one, and whether feedback still matches the current state of the work.

In this setup, the handoff was much cleaner. The planner produced the design, the reviewer responded to that design, and the workflow moved forward from a shared version of the task rather than from several disconnected conversations.

Another practical advantage was that the process felt more like managing a small engineering workflow than prompting isolated tools. That does not mean the agents replaced judgment or engineering decision-making. It means the structure around the task was better. The workflow encouraged clearer stages, clearer deliverables, and a better point at which to move from design into code.

What I also liked is that the setup matched the nature of the feature. JWT authentication and role-based access control are not huge systems, but they do involve enough moving parts that planning and review genuinely matter. This made the workflow feel justified. It did not feel like multiple agents were being used just for novelty. Each role had a reason to exist.

Overall, what worked well was not simply that I used Claude Code, Codex, and BAND together. What worked well was the combination of role separation, shared context, structured handoff, and reviewing the design before implementation. That is what made the workflow feel more practical than the usual copy-paste pattern between multiple AI coding tools.

---

## 9. Where Human Judgment Still Matters

Even in a workflow like this, the developer is still the one responsible for the outcome. The agents can help structure the work, challenge weak spots, and accelerate implementation, but they do not remove the need for judgment. They make the workflow more organized. They do not replace ownership.

That becomes especially important on a feature like authentication. A planner can propose a clean design, and a reviewer can identify missing pieces, but neither of them fully understands the broader product context on its own. Decisions such as how strict the token lifecycle should be, what counts as an admin action, or which trade-offs are acceptable for the current stage of the project still need human direction.

Human judgment also matters when deciding whether the plan is actually good enough to implement. A reviewed design can still be overly complex, too narrow, or misaligned with how the system is expected to evolve. The same is true during implementation. Even if the code looks reasonable, someone still needs to decide whether it fits the existing codebase, whether the abstractions are appropriate, and whether the feature is ready to ship.

There is also a practical point here: shared workflows reduce coordination overhead, but they do not eliminate ambiguity. Agents can misunderstand constraints, over-assume what the repo already provides, or push the design in a direction that is technically valid but not the one you want. That means the developer still has to supervise the process, not at the level of copying outputs between tools, but at the level of approving direction and making trade-offs.

That is actually the version of AI-assisted development I find more useful. The goal is not to disappear from the loop. The goal is to spend less time acting as middleware between tools and more time acting like the engineer responsible for the system. In this workflow, BAND helped reduce the mechanical overhead of coordination, but the important decisions still stayed where they should: with the developer.

Using multiple coding agents is not difficult anymore. Many developers already do it. The harder part is making those agents work together without turning yourself into the manual coordination layer between them.

That is what made this workflow interesting. Instead of treating Claude Code and Codex as separate tools that needed constant copy-paste and context syncing, BAND made it possible to run them in a more connected way. Claude Code handled the planning, Codex handled the review, and the workflow moved from design to implementation through shared artifacts and cleaner handoffs.

For this FastAPI feature, that structure made the biggest difference. Building JWT authentication and role-based access control is not an enormous project, but it has enough moving parts that planning and review genuinely matter. Starting implementation from a reviewed plan made the process feel more focused and less fragile than jumping directly from a prompt into code.

I do not think the value here is simply “more agents.” The value is better coordination. When planning, reviewing, and implementation are treated as connected stages rather than isolated conversations, the workflow starts to feel much closer to a real engineering process.

That is the main takeaway from this experiment. BAND did not remove the need for judgment, and it did not magically solve software design. What it did do was reduce the friction of working with multiple coding agents and make the path from plan to implementation feel much more practical.
